# 1. 准备原始 CapgMyo DB-a 数据

下载并校验 Figshare v1 的 18 个压缩包，转换为 `.pt` 并划分 train/val/test。

In [ ]:
import hashlib
import io
import json
import os
import re
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import requests
import torch
from requests.adapters import HTTPAdapter
from scipy.io import loadmat
from tqdm.auto import tqdm
from urllib3.util.retry import Retry


def find_project_root():
    # 允许从项目根目录或 notebook 所在目录启动 Jupyter。
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'data_prepare').is_dir() and (candidate / 'CapgMyo_data').is_dir():
            return candidate
    raise RuntimeError('找不到项目根目录：应同时包含 src/data_prepare 和 CapgMyo_data。')


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'CapgMyo_data' / 'raw'
DOWNLOAD_DIR = RAW_DIR / 'downloads'
SPLIT_DIRS = {name: RAW_DIR / name for name in ('train', 'val', 'test')}
MANIFEST_PATH = RAW_DIR / 'split_manifest.json'
METADATA_PATH = DOWNLOAD_DIR / 'figshare_article_7210397_v1.json'

ARTICLE_ID = 7210397
ARTICLE_VERSION = 1
ARTICLE_API_URL = f'https://api.figshare.com/v2/articles/{ARTICLE_ID}/versions/{ARTICLE_VERSION}'
ARTICLE_URL = 'https://figshare.com/articles/dataset/7210397'
ARTICLE_DOI = '10.6084/m9.figshare.7210397.v1'
SPLIT_SEED = 42
OVERWRITE_PT = False
USE_ENV_PROXY = os.getenv('CAPGMYO_USE_ENV_PROXY', '').strip().lower() in {'1', 'true', 'yes'}
EXPECTED_TRIAL_SHAPE = (1000, 1, 8, 16)
EXPECTED_COUNTS = {'train': 1008, 'val': 144, 'test': 288}

# 固定 Figshare v1 的文件身份，防止上游文件变化后静默得到另一份数据。
PINNED_ARCHIVES = [
    {'name': 'dba-s1.zip', 'id': 13277105, 'size': 78413147, 'md5': 'ee30c5235d817e3c96492aff01fc9423'},
    {'name': 'dba-s2.zip', 'id': 13275896, 'size': 78482918, 'md5': '63a0dbf3f6387097d957a9cdca2182c9'},
    {'name': 'dba-s3.zip', 'id': 13275953, 'size': 78342973, 'md5': 'cef147a5a0a0be71f9da30d15461dbb3'},
    {'name': 'dba-s4.zip', 'id': 13275962, 'size': 78178750, 'md5': '21bf535d25369b1c1cda33d582105df8'},
    {'name': 'dba-s5.zip', 'id': 13275965, 'size': 78434204, 'md5': '397a6fd60c7c63e5a6e6254957435b35'},
    {'name': 'dba-s6.zip', 'id': 13275983, 'size': 78160828, 'md5': '9e16e3686e9620da3d6be980b7b1e4e0'},
    {'name': 'dba-s7.zip', 'id': 13276016, 'size': 78419193, 'md5': '0689e978553f3ba1a32d124fca097fad'},
    {'name': 'dba-s8.zip', 'id': 13276019, 'size': 78374934, 'md5': '6bc3a96a4312dab93b1a198df2c63b59'},
    {'name': 'dba-s9.zip', 'id': 13276022, 'size': 78306564, 'md5': '743deeded248e00c0faee2d86c4b3e6d'},
    {'name': 'dba-s10.zip', 'id': 13276934, 'size': 78437109, 'md5': 'c8fe5aa7e61d44d25ac086852123aa60'},
    {'name': 'dba-s11.zip', 'id': 13277147, 'size': 78389841, 'md5': '74382cc1927835c731b1305d4a3f6501'},
    {'name': 'dba-s12.zip', 'id': 13277000, 'size': 78435013, 'md5': 'ec92802ea6a9e656805895fc8b5016f9'},
    {'name': 'dba-s13.zip', 'id': 13277015, 'size': 78431118, 'md5': 'a29e85b204168fccb8104922ca9c01ae'},
    {'name': 'dba-s14.zip', 'id': 13277027, 'size': 78429745, 'md5': '574845abe6de997158f82f19d6ab03cb'},
    {'name': 'dba-s15.zip', 'id': 13276136, 'size': 78140359, 'md5': 'c1e85885f2c14dd41e22de90ab771a29'},
    {'name': 'dba-s16.zip', 'id': 13276139, 'size': 78438262, 'md5': 'c742e7d0f2b45bbe45b1f7e5bb5bf9c5'},
    {'name': 'dba-s17.zip', 'id': 13276142, 'size': 77926359, 'md5': 'c638a7ffe355be522b1f0f4e4e1bb799'},
    {'name': 'dba-s18.zip', 'id': 13276148, 'size': 78138642, 'md5': '386a49bf4f6a8c86135d259d7a26d591'},
]

# 用固定种子生成一次重复编号排列，并锁定结果以便跨环境复现。
repetition_permutation = np.random.default_rng(SPLIT_SEED).permutation(np.arange(1, 11)).tolist()
assert repetition_permutation == [6, 7, 1, 8, 4, 3, 5, 10, 2, 9]
SPLIT_REPETITIONS = {
    'train': sorted(repetition_permutation[:7]),
    'val': sorted(repetition_permutation[7:8]),
    'test': sorted(repetition_permutation[8:]),
}


def build_http_session():
    session = requests.Session()
    # Windows/Conda 可能继承不可用的系统代理并触发 TLS EOF；默认直接连接。
    session.trust_env = USE_ENV_PROXY
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({'GET'}),
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    session.headers.update({'User-Agent': 'CapgMyo-data-preparation/1.0'})
    return session


HTTP_SESSION = build_http_session()

for directory in (RAW_DIR, DOWNLOAD_DIR, *SPLIT_DIRS.values()):
    directory.mkdir(parents=True, exist_ok=True)

print(f'项目根目录：{PROJECT_ROOT}')
print(f'原始数据目录：{RAW_DIR}')
print(f'重复次数划分：{SPLIT_REPETITIONS}')

In [ ]:
def write_json_atomic(path, value):
    # 临时文件写完后再替换，避免 notebook 中断时留下半个 JSON。
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def md5sum(path):
    digest = hashlib.md5()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def build_download_urls(file_id, preferred_url=None):
    # 两个域名都是 Figshare 官方下载入口，可绕开部分云出口的单域名封禁。
    candidates = [
        preferred_url,
        f'https://ndownloader.figshare.com/files/{file_id}',
        f'https://figshare.com/ndownloader/files/{file_id}',
    ]
    return list(dict.fromkeys(url for url in candidates if url))


def build_pinned_article():
    # API 不可达时只补足后续流程所需字段；文件身份仍由固定 size 和 MD5 校验。
    files = []
    for pinned in PINNED_ARCHIVES:
        files.append({
            'id': pinned['id'],
            'name': pinned['name'],
            'size': pinned['size'],
            'computed_md5': pinned['md5'],
            'download_url': build_download_urls(pinned['id'])[0],
        })
    return {
        'id': ARTICLE_ID,
        'version': ARTICLE_VERSION,
        'doi': ARTICLE_DOI,
        'figshare_url': ARTICLE_URL,
        'license': {
            'value': 2,
            'name': 'CC0',
            'url': 'https://creativecommons.org/publicdomain/zero/1.0/',
        },
        'files': files,
        '_metadata_source': 'notebook-pinned-fallback',
    }


def validate_article_metadata(article):
    if article.get('id') != ARTICLE_ID or article.get('version') != ARTICLE_VERSION:
        raise RuntimeError('Figshare 返回的文章 ID 或版本与锁定版本不一致。')

    remote_files = {item['name']: item for item in article.get('files', [])}
    resolved_files = []
    for pinned in PINNED_ARCHIVES:
        remote = remote_files.get(pinned['name'])
        if remote is None:
            raise RuntimeError(f"Figshare 缺少锁定文件：{pinned['name']}")
        remote_md5 = remote.get('computed_md5') or remote.get('supplied_md5')
        observed = (remote.get('id'), remote.get('size'), remote_md5)
        expected = (pinned['id'], pinned['size'], pinned['md5'])
        if observed != expected:
            raise RuntimeError(
                f"上游文件信息发生变化：{pinned['name']}，"
                f"期望 {expected}，实际 {observed}。"
            )
        resolved = dict(pinned)
        resolved['download_urls'] = build_download_urls(
            pinned['id'], remote.get('download_url')
        )
        resolved['download_url'] = resolved['download_urls'][0]
        resolved_files.append(resolved)

    if set(remote_files) != {item['name'] for item in PINNED_ARCHIVES}:
        raise RuntimeError('Figshare v1 的文件集合与 notebook 锁定清单不一致。')
    return resolved_files


def fetch_and_validate_metadata():
    try:
        response = HTTP_SESSION.get(ARTICLE_API_URL, timeout=(30, 120))
        response.raise_for_status()
        article = response.json()
        metadata_source = 'Figshare API'
    except (requests.RequestException, ValueError) as error:
        # Figshare/WAF 可能拒绝数据中心 IP；缓存不存在时使用已锁定的官方清单。
        if METADATA_PATH.is_file():
            try:
                article = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
            except (OSError, ValueError) as cache_error:
                raise RuntimeError(f'Figshare API 不可用，且元数据缓存损坏：{METADATA_PATH}') from cache_error
            metadata_source = f'本地缓存 {METADATA_PATH}'
        else:
            article = build_pinned_article()
            metadata_source = 'notebook 锁定清单'
        print(f'警告：Figshare API 请求失败（{error}），改用{metadata_source}。')

    resolved_files = validate_article_metadata(article)
    if metadata_source == 'Figshare API':
        write_json_atomic(METADATA_PATH, article)
    return article, resolved_files


def archive_is_valid(path, metadata):
    return (
        path.is_file()
        and path.stat().st_size == metadata['size']
        and md5sum(path) == metadata['md5']
    )


def request_archive_stream(metadata, headers):
    errors = []
    download_urls = metadata.get('download_urls') or [metadata['download_url']]
    for url in download_urls:
        response = None
        try:
            response = HTTP_SESSION.get(
                url, headers=headers, stream=True, timeout=(30, 120)
            )
            response.raise_for_status()
            return response, url
        except requests.RequestException as error:
            if response is not None:
                response.close()
            errors.append(f'{url}: {error}')

    details = '\n'.join(errors)
    raise RuntimeError(
        f"无法从 Figshare 下载 {metadata['name']}，所有官方直链均失败。\n"
        '远程云主机的出口 IP 可能被 Figshare/WAF 拒绝。请在可访问 Figshare 的本机'
        f'下载该文件并上传到 {DOWNLOAD_DIR}；或配置 HTTPS_PROXY，并设置 '
        f'CAPGMYO_USE_ENV_PROXY=1 后重新运行第一个代码单元。\n{details}'
    )


def download_archive(metadata, progress):
    destination = DOWNLOAD_DIR / metadata['name']
    partial_path = destination.with_suffix(destination.suffix + '.part')

    if archive_is_valid(destination, metadata):
        return destination

    resume_from = partial_path.stat().st_size if partial_path.exists() else 0
    if resume_from > metadata['size']:
        partial_path.unlink()
        progress.update(-resume_from)
        resume_from = 0
    elif resume_from == metadata['size']:
        if md5sum(partial_path) == metadata['md5']:
            os.replace(partial_path, destination)
            return destination
        partial_path.unlink()
        progress.update(-resume_from)
        resume_from = 0

    headers = {'Range': f'bytes={resume_from}-'} if resume_from else {}
    response, selected_url = request_archive_stream(metadata, headers)

    # 某些镜像会忽略 Range；此时从头写入并回退进度，避免重复拼接。
    if resume_from and response.status_code != 206:
        progress.update(-resume_from)
        resume_from = 0

    mode = 'ab' if resume_from else 'wb'
    progress.set_postfix_str(f"{metadata['name']} @ {selected_url.split('/')[2]}")
    try:
        with partial_path.open(mode) as file_handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                file_handle.write(chunk)
                progress.update(len(chunk))
    finally:
        response.close()

    actual_size = partial_path.stat().st_size
    actual_md5 = md5sum(partial_path)
    if actual_size != metadata['size'] or actual_md5 != metadata['md5']:
        # 不保留已确认损坏的临时文件，下一次运行会重新下载该压缩包。
        progress.update(-actual_size)
        partial_path.unlink()
        raise RuntimeError(
            f"下载校验失败：{metadata['name']}；"
            f"size={actual_size}，md5={actual_md5}"
        )

    os.replace(partial_path, destination)
    return destination


def download_all_archives(resolved_files):
    states = []
    initial_bytes = 0
    for metadata in resolved_files:
        destination = DOWNLOAD_DIR / metadata['name']
        partial_path = destination.with_suffix(destination.suffix + '.part')
        valid = archive_is_valid(destination, metadata)
        partial_size = min(partial_path.stat().st_size, metadata['size']) if partial_path.exists() else 0
        initial_bytes += metadata['size'] if valid else partial_size
        states.append((metadata, valid))

    total_bytes = sum(item['size'] for item in resolved_files)
    archive_paths = []
    with tqdm(
        total=total_bytes,
        initial=initial_bytes,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
        desc='下载 CapgMyo DB-a',
    ) as progress:
        for metadata, valid in states:
            if valid:
                progress.set_postfix_str(f"已校验 {metadata['name']}")
                archive_paths.append(DOWNLOAD_DIR / metadata['name'])
            else:
                archive_paths.append(download_archive(metadata, progress))

    return archive_paths

In [ ]:
def subject_id_from_archive(path):
    match = re.fullmatch(r'dba-s(\d+)\.zip', path.name, flags=re.IGNORECASE)
    if match is None:
        raise ValueError(f'无法从压缩包名解析受试者编号：{path.name}')
    return int(match.group(1))


def normalize_index_set(values, expected_count, field_name):
    observed = set(values)
    one_based = set(range(1, expected_count + 1))
    zero_based = set(range(expected_count))
    if observed == one_based:
        return {value: value for value in observed}
    if observed == zero_based:
        return {value: value + 1 for value in observed}
    raise ValueError(
        f'{field_name}编号集合异常：{sorted(observed)}；'
        f'期望 1～{expected_count} 或 0～{expected_count - 1}。'
    )


def enumerate_trial_members(zip_file, subject_id):
    mat_members = [
        info.filename
        for info in zip_file.infolist()
        if not info.is_dir() and info.filename.lower().endswith('.mat')
    ]
    if len(mat_members) != 80:
        raise ValueError(
            f'受试者 {subject_id} 的 .mat 文件数为 {len(mat_members)}，期望 80。'
        )

    raw_entries = []
    for member in mat_members:
        # 官方文件名末尾的两个数字分别表示手势和重复次数；兼容 - 与 _ 分隔。
        numbers = [int(value) for value in re.findall(r'\d+', member)]
        if len(numbers) < 2:
            raise ValueError(f'无法从文件名解析手势和重复编号：{member}')
        raw_entries.append((member, numbers[-2], numbers[-1]))

    gesture_map = normalize_index_set([item[1] for item in raw_entries], 8, '手势')
    repetition_map = normalize_index_set([item[2] for item in raw_entries], 10, '重复次数')
    entries = [
        {
            'member': member,
            'gesture_id': gesture_map[raw_gesture],
            'repetition_id': repetition_map[raw_repetition],
        }
        for member, raw_gesture, raw_repetition in raw_entries
    ]

    keys = [(item['gesture_id'], item['repetition_id']) for item in entries]
    expected_keys = {(gesture, repetition) for gesture in range(1, 9) for repetition in range(1, 11)}
    if len(keys) != len(set(keys)) or set(keys) != expected_keys:
        raise ValueError(f'受试者 {subject_id} 的手势-重复组合不完整或存在重复。')

    return sorted(entries, key=lambda item: (item['gesture_id'], item['repetition_id']))


def load_trial_tensor(zip_file, member):
    try:
        matlab = loadmat(io.BytesIO(zip_file.read(member)), variable_names=['data'])
    except NotImplementedError as error:
        raise RuntimeError(f'{member} 使用了当前 SciPy 不支持的 MATLAB 格式。') from error

    if 'data' not in matlab:
        raise KeyError(f'{member} 中不存在官方数据字段 data。')

    data = np.asarray(matlab['data']).squeeze()
    if data.shape == (128, 1000):
        data = data.T
    if data.shape != (1000, 128):
        raise ValueError(f'{member} 的 data shape 为 {data.shape}，期望 (1000, 128)。')
    if not np.issubdtype(data.dtype, np.number):
        raise TypeError(f'{member} 的 data 不是数值数组：{data.dtype}')
    if not np.isfinite(data).all():
        raise ValueError(f'{member} 的 data 包含 NaN 或 Inf。')

    # 只改变 dtype 和视图形状，不执行归一化、裁剪或整流。
    data = np.asarray(data, dtype=np.float32, order='C').reshape(EXPECTED_TRIAL_SHAPE)
    return torch.from_numpy(data)


def split_for_repetition(repetition_id):
    for split_name, repetitions in SPLIT_REPETITIONS.items():
        if repetition_id in repetitions:
            return split_name
    raise ValueError(f'重复编号不属于任何划分：{repetition_id}')


def load_saved_pt(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        # 兼容尚未提供 weights_only 参数的旧版 PyTorch。
        return torch.load(path, map_location='cpu')


def saved_pt_is_valid(path, subject_id, gesture_id, repetition_id):
    if not path.is_file():
        return False
    try:
        payload = load_saved_pt(path)
        return (
            isinstance(payload, dict)
            and isinstance(payload.get('data'), torch.Tensor)
            and tuple(payload['data'].shape) == EXPECTED_TRIAL_SHAPE
            and payload['data'].dtype == torch.float32
            and int(payload['label']) == gesture_id - 1
            and int(payload['subject_id']) == subject_id
            and int(payload['gesture_id']) == gesture_id
            and int(payload['repetition_id']) == repetition_id
        )
    except (EOFError, KeyError, OSError, RuntimeError, TypeError, ValueError):
        return False


def save_trial_atomic(path, payload):
    # 同一文件系统内原子替换，避免中断后把不完整文件当成有效样本。
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def convert_archives(archive_paths):
    records = []
    counts = Counter()
    expected_total = sum(EXPECTED_COUNTS.values())

    with tqdm(total=expected_total, unit='trial', desc='转换为 .pt') as progress:
        for archive_path in archive_paths:
            subject_id = subject_id_from_archive(archive_path)
            with zipfile.ZipFile(archive_path) as zip_file:
                # 压缩包已通过固定 MD5 校验，无需再解压全部成员做重复 CRC 扫描。
                entries = enumerate_trial_members(zip_file, subject_id)

                for entry in entries:
                    gesture_id = entry['gesture_id']
                    repetition_id = entry['repetition_id']
                    split_name = split_for_repetition(repetition_id)
                    subject_dir = SPLIT_DIRS[split_name] / f'subject_{subject_id:02d}'
                    subject_dir.mkdir(parents=True, exist_ok=True)
                    output_path = subject_dir / (
                        f's{subject_id:02d}_g{gesture_id:02d}_r{repetition_id:02d}.pt'
                    )

                    valid_existing = (
                        not OVERWRITE_PT
                        and saved_pt_is_valid(
                            output_path, subject_id, gesture_id, repetition_id
                        )
                    )
                    if not valid_existing:
                        tensor = load_trial_tensor(zip_file, entry['member'])
                        payload = {
                            'data': tensor,
                            'label': torch.tensor(gesture_id - 1, dtype=torch.int64),
                            'subject_id': torch.tensor(subject_id, dtype=torch.int64),
                            'gesture_id': torch.tensor(gesture_id, dtype=torch.int64),
                            'repetition_id': torch.tensor(repetition_id, dtype=torch.int64),
                        }
                        save_trial_atomic(output_path, payload)

                    records.append({
                        'path': output_path.relative_to(PROJECT_ROOT).as_posix(),
                        'split': split_name,
                        'subject_id': subject_id,
                        'gesture_id': gesture_id,
                        'label': gesture_id - 1,
                        'repetition_id': repetition_id,
                        'source_archive': archive_path.name,
                        'source_member': entry['member'],
                        'shape': list(EXPECTED_TRIAL_SHAPE),
                        'dtype': 'torch.float32',
                    })
                    counts[split_name] += 1
                    progress.update(1)
                    progress.set_postfix(subject=subject_id, split=split_name)

    if dict(counts) != EXPECTED_COUNTS:
        raise RuntimeError(f'划分数量异常：{dict(counts)}，期望 {EXPECTED_COUNTS}。')
    return records, dict(counts)


def build_manifest(article, resolved_files, records, counts):
    manifest = {
        'format_version': 1,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'dataset': 'CapgMyo DB-a official preprocessed data',
        'source': {
            'article_id': ARTICLE_ID,
            'article_version': ARTICLE_VERSION,
            'doi': ARTICLE_DOI,
            'article_url': article.get('figshare_url', ARTICLE_URL),
            'api_url': ARTICLE_API_URL,
            'license': article.get('license', {}).get('name'),
            'archives': resolved_files,
        },
        'split': {
            'strategy': 'within-subject split by repetition id',
            'seed': SPLIT_SEED,
            'rng': 'numpy.random.default_rng',
            'permutation': repetition_permutation,
            'repetitions': SPLIT_REPETITIONS,
            'counts': counts,
        },
        'sample': {
            'data_shape': list(EXPECTED_TRIAL_SHAPE),
            'data_dtype': 'torch.float32',
            'label_dtype': 'torch.int64',
            'gesture_id_range': [1, 8],
            'label_range': [0, 7],
            'subject_id_range': [1, 18],
            'repetition_id_range': [1, 10],
        },
        'records': records,
    }
    write_json_atomic(MANIFEST_PATH, manifest)
    return manifest

In [ ]:
article_metadata, resolved_archives = fetch_and_validate_metadata()
archive_paths = download_all_archives(resolved_archives)
print(f'已准备 {len(archive_paths)} 个官方压缩包。')

In [ ]:
records, split_counts = convert_archives(archive_paths)
manifest = build_manifest(
    article_metadata, resolved_archives, records, split_counts
)
print('数据准备完成。')
print(f'划分数量：{split_counts}')
print(f'划分清单：{MANIFEST_PATH}')

In [ ]:
observed_counts = {
    split_name: len(list(split_dir.glob('subject_*/*.pt')))
    for split_name, split_dir in SPLIT_DIRS.items()
}
if observed_counts != EXPECTED_COUNTS:
    raise RuntimeError(f'磁盘上的 .pt 数量异常：{observed_counts}')

example_path = next(SPLIT_DIRS['train'].glob('subject_*/*.pt'))
example = load_saved_pt(example_path)
print(f'文件数量：{observed_counts}')
print(f'样例文件：{example_path.relative_to(PROJECT_ROOT)}')
print({
    'data_shape': tuple(example['data'].shape),
    'data_dtype': example['data'].dtype,
    'label': int(example['label']),
    'subject_id': int(example['subject_id']),
    'gesture_id': int(example['gesture_id']),
    'repetition_id': int(example['repetition_id']),
})